In [12]:
'''
PyTorch 1.0 强化学习 (Packt Publishing)
第2章：马尔可夫决策过程与动态规划
作者：Yuxi (Hayden) Liu
修改说明：升级至 gymnasium 库 (FrozenLake-v1)，增加详细中文注释，优化中文输出与策略可视化
'''

import torch
import gymnasium as gym  # 使用新版 gymnasium 库

# ==========================================
# 1. 初始化环境与超参数
# ==========================================

# 创建 FrozenLake-v1 环境
# is_slippery=True 表示冰面很滑，智能体执行动作后可能会向垂直方向滑动（增加难度）
env = gym.make('FrozenLake-v1', is_slippery=True)

gamma = 0.99       # 折扣因子：越接近1，越看重未来的长期奖励
threshold = 0.0001 # 收敛阈值：当价值变化小于此值时，停止策略评估迭代

In [13]:
# ==========================================
# 2. 核心算法：策略评估 (Policy Evaluation)
# ==========================================

def policy_evaluation(env, policy, gamma, threshold):
    """
    执行策略评估：计算给定策略下的状态价值函数 V
    原理：通过迭代应用贝尔曼期望方程，直到价值函数收敛
    
    参数:
        env: Gymnasium 环境实例
        policy: 策略向量，一维张量，表示在每个状态下选择的确定性动作索引
        gamma: 折扣因子
        threshold: 收敛阈值
    返回:
        V: 收敛后的状态价值向量
    """
    n_state = policy.shape[0]  # 获取状态总数 (4x4地图为16)
    V = torch.zeros(n_state)   # 初始化所有状态的价值为 0
    
    while True:
        V_temp = torch.zeros(n_state)  # 临时存储本次迭代计算出的新价值
        
        for state in range(n_state):
            # 获取当前策略在该状态下决定的动作
            action = policy[state].item()
            
            # 遍历该 (状态, 动作) 对的所有可能转移结果
            # env.unwrapped.P[state][action] 返回一个列表，包含元组: (转移概率, 下一状态, 奖励, 是否结束)
            for trans_prob, new_state, reward, _ in env.unwrapped.P[state][action]:
                # 贝尔曼期望方程：V(s) = Σ P(s'|s,a) * [R + γ * V(s')]
                V_temp[state] += trans_prob * (reward + gamma * V[new_state])
        
        # 计算本次迭代与上一次迭代之间，所有状态价值的最大变化量
        max_delta = torch.max(torch.abs(V - V_temp))
        
        # 更新价值函数
        V = V_temp.clone()
        
        # 如果最大变化量小于阈值，说明已经收敛，跳出循环
        if max_delta <= threshold:
            break
            
    return V

In [14]:
# ==========================================
# 3. 核心算法：策略改进 (Policy Improvement)
# ==========================================

def policy_improvement(env, V, gamma):
    """
    执行策略改进：基于当前的状态价值 V，贪心地选择能带来最大期望价值的动作，生成新策略
    原理：利用当前价值函数评估所有动作的优劣，选择最优动作（贪心策略）
    
    参数:
        env: Gymnasium 环境实例
        V: 当前策略的状态价值向量
        gamma: 折扣因子
    返回:
        policy: 改进后的新策略向量
    """
    n_state = env.observation_space.n  # 状态数
    n_action = env.action_space.n      # 动作数 (左、下、右、上)
    policy = torch.zeros(n_state)      # 初始化新策略
    
    for state in range(n_state):
        v_actions = torch.zeros(n_action)  # 存储该状态下执行各个动作的期望价值
        
        for action in range(n_action):
            # 计算执行该动作的期望价值：Q(s,a) = Σ P(s'|s,a) * [R + γ * V(s')]
            for trans_prob, new_state, reward, _ in env.unwrapped.P[state][action]:
                v_actions[action] += trans_prob * (reward + gamma * V[new_state])
                
        # 贪心策略：选择期望价值最大的动作作为该状态下的新动作
        policy[state] = torch.argmax(v_actions)
        
    return policy

In [15]:
# ==========================================
# 4. 核心算法：策略迭代 (Policy Iteration)
# ==========================================

def policy_iteration(env, gamma, threshold):
    """
    执行策略迭代算法求解环境
    原理：交替执行“策略评估”和“策略改进”，直到策略不再变化（收敛到最优策略）
    
    参数:
        env: Gymnasium 环境实例
        gamma: 折扣因子
        threshold: 收敛阈值
    返回:
        V: 最优状态价值向量
        policy: 最优策略向量
    """
    n_state = env.observation_space.n
    n_action = env.action_space.n
    
    # 初始化一个随机策略（每个状态随机分配一个动作）
    policy = torch.randint(high=n_action, size=(n_state,)).float()
    
    iteration = 0  # 记录迭代次数
    
    while True:
        iteration += 1
        # 步骤 1：策略评估 —— 计算当前策略的价值函数
        V = policy_evaluation(env, policy, gamma, threshold)
        
        # 步骤 2：策略改进 —— 基于价值函数提升策略
        policy_improved = policy_improvement(env, V, gamma)
        
        # 检查策略是否已经收敛（新旧策略完全一致）
        if torch.equal(policy_improved, policy):
            print(f"  [收敛] 策略迭代在第 {iteration} 次循环后收敛。")
            return V, policy_improved
            
        # 更新策略，进入下一轮迭代
        policy = policy_improved

In [16]:
# ==========================================
# 5. 执行策略迭代并打印结果
# ==========================================

print("=" * 60)
print("【开始执行策略迭代算法 (Policy Iteration)】")
print("=" * 60)

V_optimal, optimal_policy = policy_iteration(env, gamma, threshold)

# 打印最优价值函数
print("\n【最优状态价值函数 V*(s)】")
# 将一维的价值向量重塑为 4x4 的网格，方便对应地图位置
V_grid = V_optimal.numpy().reshape(4, 4)
for row in V_grid:
    print("  [" + ", ".join(f"{val:.4f}" for val in row) + "]")

# 打印最优策略（转换为直观的箭头符号）
print("\n【最优策略 π*(s)】(箭头代表智能体应该采取的动作方向)")
action_symbols = ['← (左)', '↓ (下)', '→ (右)', '↑ (上)']
policy_grid = optimal_policy.numpy().astype(int).reshape(4, 4)
for row in policy_grid:
    print("  [" + ", ".join(action_symbols[action] for action in row) + "]")

print("\n【说明】")
print("  S = 起点 (Start), F = 冰面 (Frozen), H = 冰洞 (Hole), G = 终点 (Goal)")
print("  智能体会根据上述箭头指示的方向移动，以最大概率避开冰洞并到达终点。")

# 关闭环境释放资源
env.close()

【开始执行策略迭代算法 (Policy Iteration)】
  [收敛] 策略迭代在第 2 次循环后收敛。

【最优状态价值函数 V*(s)】
  [0.5404, 0.4966, 0.4681, 0.4541]
  [0.5569, 0.0000, 0.3572, 0.0000]
  [0.5905, 0.6421, 0.6144, 0.0000]
  [0.0000, 0.7410, 0.8625, 0.0000]

【最优策略 π*(s)】(箭头代表智能体应该采取的动作方向)
  [← (左), ↑ (上), ↑ (上), ↑ (上)]
  [← (左), ← (左), ← (左), ← (左)]
  [↑ (上), ↓ (下), ← (左), ← (左)]
  [← (左), → (右), ↓ (下), ← (左)]

【说明】
  S = 起点 (Start), F = 冰面 (Frozen), H = 冰洞 (Hole), G = 终点 (Goal)
  智能体会根据上述箭头指示的方向移动，以最大概率避开冰洞并到达终点。


这段代码的核心逻辑非常清晰，它是强化学习中 **策略迭代（Policy Iteration）** 的标准实现，专门用于解决已知环境模型（Model-based）的马尔可夫决策过程（MDP）。

简单来说，它的任务是：**通过不断交替执行“评估当前策略”和“改进当前策略”两个步骤，最终找到一个能让智能体获得最大长期收益的完美行为模式。**

下面我按照代码从上到下的顺序，拆解它**讲了什么（目的）**以及**怎么做的（实现原理）**：

---

### 第一部分：初始化环境与超参数（第 1-20 行）
**讲了什么：** 搭建一个 4x4 的虚拟“冰湖”世界，并设定算法运行的基本规则。
**怎么做的：**
1.  **创建环境**：使用新版 `gymnasium` 库创建 `FrozenLake-v1` 环境。设定 `is_slippery=True`，这意味着冰面很滑，智能体执行动作后可能会向垂直方向随机滑动，增加了环境的随机性。
2.  **定义超参数**：
    *   `gamma = 0.99`：折扣因子，非常接近 1，代表智能体非常看重未来的长期奖励。
    *   `threshold = 0.0001`：收敛阈值，用于判断策略评估是否已经稳定。

---

### 第二部分：核心算法 - 策略评估 `policy_evaluation`（第 22-56 行）
**讲了什么：** 给定一个固定的行为模式（策略），计算在这个模式下，每个状态的“长期价值”是多少。
**怎么做的：**
1.  **初始化**：把所有状态的初始价值 $V$ 设为 0。
2.  **进入 `while True` 循环（迭代更新）**：
    *   **遍历状态与动作**：遍历每一个状态 `state`，获取当前策略在该状态下决定的动作 `action`。
    *   **核心公式计算（贝尔曼期望方程）**：
        ```python
        V_temp[state] += trans_prob * (reward + gamma * V[new_state])
        ```
        这行代码完美对应了数学公式：
        $$V(s) \leftarrow \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V(s') \right]$$
        *   通过 `env.unwrapped.P[state][action]` 获取该 (状态, 动作) 对的所有可能转移结果（转移概率、下一状态、奖励）。
        *   将每种可能性的“即时奖励 + 折扣后的未来价值”乘以转移概率，累加起来就是新的状态价值。
    *   **检查收敛**：计算新旧价值的最大差值 `max_delta`。如果差值小于 `threshold`，说明价值已经稳定，跳出循环；否则继续下一轮迭代。

---

### 第三部分：核心算法 - 策略改进 `policy_improvement`（第 58-82 行）
**讲了什么：** 看着刚刚算出来的状态价值，贪心地选择能让价值最大的动作，从而生成一个更好的新策略。
**怎么做的：**
1.  **遍历所有状态**：对于每一个状态 `state`，准备计算所有可能动作的价值。
2.  **计算动作价值（Q值）**：
    *   遍历该状态下的所有动作 `action`。
    *   同样利用环境模型 `env.unwrapped.P`，计算执行该动作的期望价值：
        $$Q(s,a) = \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V(s') \right]$$
3.  **贪心选择**：使用 `torch.argmax(v_actions)` 找到期望价值最大的动作，将其作为该状态下的新动作。这就构成了一个确定性的贪心策略。

---

### 第四部分：核心算法 - 策略迭代 `policy_iteration`（第 84-114 行）
**讲了什么：** 这是代码的灵魂。它将“策略评估”和“策略改进”组合起来，形成完整的迭代闭环，直到找到最优解。
**怎么做的：**
1.  **初始化策略**：使用 `torch.randint` 生成一个完全随机的初始策略。
2.  **进入 `while True` 循环**：
    *   **步骤 1**：调用 `policy_evaluation`，计算当前策略的价值函数 $V$。
    *   **步骤 2**：调用 `policy_improvement`，基于价值函数 $V$ 提升策略，得到新策略。
    *   **检查收敛**：使用 `torch.equal` 检查新旧策略是否完全一致。如果一致，说明策略已经无法再改进，达到了最优状态，返回最终的价值和策略；否则，用新策略替换旧策略，进入下一轮迭代。

---

### 第五部分：执行算法并打印结果（第 116-末尾）
**讲了什么：** 运行策略迭代算法，并将枯燥的数字矩阵转换为直观的可视化结果。
**怎么做的：**
1.  **执行算法**：调用 `policy_iteration` 获取最优价值 `V_optimal` 和最优策略 `optimal_policy`。
2.  **价值可视化**：将一维的最优价值向量重塑（`reshape`）为 4x4 的网格，按行打印，方便对应地图上的实际位置。
3.  **策略可视化**：
    *   定义动作符号映射：`['← (左)', '↓ (下)', '→ (右)', '↑ (上)']`。
    *   将一维的最优策略向量重塑为 4x4 网格。
    *   遍历网格，将动作索引转换为对应的箭头符号打印出来，形成一张清晰的“逃生路线图”。
4.  **清理资源**：调用 `env.close()` 关闭环境。

---

### 总结
这段代码**从上到下**完成了一个完整的动态规划求解闭环：
**定义环境与参数 → 设定评估函数 → 设定改进函数 → 循环迭代直至收敛 → 可视化输出最优解**。

它充分利用了 FrozenLake 环境的已知模型（通过 `env.unwrapped.P` 获取转移概率），通过数学计算直接“算”出了最优策略，而不是通过试错来“学”。
